# Module 16: Did Something Change?

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Everything in this series has been building to this question, and it is the one
public safety data gets asked most often. A programme launched, a policy
changed, a commander arrived. Did it work?

This notebook answers it three ways on the same data, against an effect whose
true size is known, and shows what each answer is worth.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")

PROGRAM_START = "2023-07"
SETTLED = "2023-11"        # the month the programme was fully in place
PRE_END = "2023-06"

ALL_TREATED = ["A001", "A002", "A004", "A007", "A010"]
VIOLATOR = "A007"          # was already improving far faster than everyone else
TREATED = [a for a in ALL_TREATED if a != VIOLATOR]

clean = final.copy()
clean = clean[~((clean["agency_id"] == "A002") &
                (clean["year_month"] == "2021-06"))]     # documented unrest
CONTROL = [a for a in clean["agency_id"].unique() if a not in ALL_TREATED]

print(f"trained and usable: {TREATED}")
print(f"comparison group  : {CONTROL}")

## 2. Two helpers

In [ ]:
def group_rate(ids):
    """Monthly pooled rate for a set of agencies."""
    w = clean[clean["agency_id"].isin(ids)].groupby("year_month")[["n_uof", "n_arrests"]].sum()
    return pd.Series((100 * w["n_uof"] / w["n_arrests"]).values,
                     index=pd.PeriodIndex(w.index, freq="M").to_timestamp())


def pooled(ids, lo, hi):
    """One pooled rate over a window."""
    w = clean[(clean["agency_id"].isin(ids)) & (clean["year_month"] >= lo)
              & (clean["year_month"] <= hi)]
    return 100 * w["n_uof"].sum() / w["n_arrests"].sum()


PRE = ("2021-07", PRE_END)
POST = (SETTLED, "2026-04")

treated_before, treated_after = pooled(TREATED, *PRE), pooled(TREATED, *POST)
control_before, control_after = pooled(CONTROL, *PRE), pooled(CONTROL, *POST)

print(f"trained agencies : {treated_before:.2f} before, {treated_after:.2f} after")
print(f"comparison group : {control_before:.2f} before, {control_after:.2f} after")

## 3. Answer one: before and after

The obvious calculation, and the one that appears in most reports.

In [ ]:
naive = 100 * (treated_after / treated_before - 1)
print(f"before and after, trained agencies only: {naive:+.1f} percent")

## 4. Answer two: difference in differences

Subtract what happened to agencies that did not adopt the programme. Whatever
was going on statewide was going on for them too.

In [ ]:
did = 100 * ((treated_after / treated_before) / (control_after / control_before) - 1)
print(f"difference in differences: {did:+.1f} percent")
print(f"\nthe comparison group changed by "
      f"{100 * (control_after / control_before - 1):+.1f} percent on its own")

## 5. Answer three: forecast the counterfactual

Fit a model on the pre programme data only, forecast forward, and compare what
happened against what the model expected. This uses no comparison group at all,
so it is available when no comparable agency exists.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

T = group_rate(TREATED)
pre = T.loc[:PRE_END]

model = ExponentialSmoothing(np.log(pre), trend="add", seasonal="add",
                             seasonal_periods=12,
                             initialization_method="estimated").fit()
horizon = len(T.loc[pd.Timestamp(PRE_END) + pd.offsets.MonthBegin(1):])
counterfactual = np.exp(model.forecast(horizon))

post = T.loc[SETTLED:].index
actual_mean = T.loc[post].mean()
expected_mean = counterfactual.loc[post].mean()

print(f"the model expected {expected_mean:.3f}, the agencies recorded {actual_mean:.3f}")
print(f"forecast counterfactual: {100 * (actual_mean / expected_mean - 1):+.1f} percent")

## 6. Score all three against the truth

In [ ]:
answers = pd.Series({
    "before and after, no comparison": naive,
    "difference in differences": did,
    "forecast counterfactual": 100 * (actual_mean / expected_mean - 1),
    "the truth built into the data": -12.0,
}).round(1)
answers.to_frame("estimated effect, percent")

**The before and after answer is more than double the truth.** Nothing is wrong
with its arithmetic. It is measuring the programme plus the statewide decline
of about 5 percent a year that
[GROUND_TRUTH.md](../../../Data/GROUND_TRUTH.md) built into every agency, and
it has no way to separate them.

The other two land within about a point of the truth, from completely different
directions. **Two methods that share no assumptions agreeing is worth more than
either one alone.**

## 7. What each answer requires

Neither of the good methods is free. Each rests on an assumption that can be
checked, and should be.

**Difference in differences assumes parallel trends**: that before the
programme, the two groups were moving in the same direction at the same speed.
Test it on the pre period.

In [ ]:
import statsmodels.formula.api as smf

d = clean[(clean["year_month"] <= PRE_END)
          & (clean["agency_id"].isin(TREATED + CONTROL))].copy()
d["treated"] = d["agency_id"].isin(TREATED).astype(int)
idx = pd.PeriodIndex(d["year_month"], freq="M")
d["t"] = (idx.year - 2019) * 12 + idx.month - 1

g = d.groupby(["treated", "t"])[["n_uof", "n_arrests"]].sum().reset_index()
g["rate"] = 100 * g["n_uof"] / g["n_arrests"]

fit = smf.ols("np.log(rate) ~ t * treated", data=g).fit()
lo, hi = fit.conf_int().loc["t:treated"]
per_year = lambda b: 100 * (np.exp(12 * b) - 1)

print(f"difference in the two groups' pre programme slopes: "
      f"{per_year(fit.params['t:treated']):+.2f} percent a year")
print(f"95 percent interval: [{per_year(lo):+.2f}, {per_year(hi):+.2f}]   "
      f"p = {fit.pvalues['t:treated']:.3f}")

The difference is small and the interval comfortably covers zero, so the
assumption holds for this comparison.

It only holds because Summit County was removed first. That agency was already
declining at about 12 percent a year before the programme began, three times
everyone else, and leaving it in biases the estimate.

In [ ]:
tb2, ta2 = pooled(ALL_TREATED, *PRE), pooled(ALL_TREATED, *POST)
print(f"difference in differences, Summit County left in : "
      f"{100 * ((ta2 / tb2) / (control_after / control_before) - 1):+.1f} percent")
print(f"difference in differences, Summit County removed : {did:+.1f} percent")
print("the truth: -12.0 percent")

**The forecast counterfactual assumes the pre programme pattern would have
continued.** There is no direct test of that, which is its main weakness: any
statewide shock after the programme started gets counted as programme effect.
What you can check is whether the model forecasts this series well over
stretches where nothing happened, which is
[Module 15](Module_15_Measuring_Forecast_Error.ipynb).

In [ ]:
sd = float(np.std(model.resid, ddof=1))
n = len(post)
point = np.log(actual_mean / expected_mean)
band = 1.96 * sd / np.sqrt(n)

print(f"forecast counterfactual: {100 * (np.exp(point) - 1):+.1f} percent")
print(f"rough interval: [{100 * (np.exp(point - band) - 1):+.1f}, "
      f"{100 * (np.exp(point + band) - 1):+.1f}]")
print("\nthis interval ignores uncertainty in the fitted parameters,")
print("so treat it as a lower bound on how wrong the estimate could be.")

## 8. Where this stops

Every method here answers the question "how much did the numbers move, beyond
what they would have done anyway". None of them establishes **why**.

The two good answers both rest on an assumption that cannot be verified from
the outcome data alone: that the comparison group, or the forecast, really does
represent what would have happened. Checking those assumptions properly,
knowing when they fail, and knowing what to do instead is the subject of the
[Causal Inference series](../../../Causal_Inference/).

What this module establishes is the minimum standard. **A before and after
comparison with no comparison group and no counterfactual is not evidence**,
and in this dataset it is wrong by a factor of two.

## 9. What to carry away

| Habit | Why |
|---|---|
| Never report before and after alone | it absorbs every trend that was already running |
| Build a comparison group, or forecast a counterfactual | both work, and they fail differently |
| Do both when you can | agreement between them is the strongest evidence available |
| Test parallel trends before using difference in differences | one divergent agency is enough to bias it |
| Screen the comparison group for the intervention | [Module 12](Module_12_Building_A_Peer_Benchmark_Series.ipynb) |
| Allow for the phase in period | an effect measured from month one is diluted |
| Report an interval | a point estimate implies a precision you do not have |

## Exercise

Run the before and after comparison on a **single** trained agency, Millgate,
and then the difference in differences. How much does the answer move?

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A004"

if AGENCY:
    ab, aa = pooled([AGENCY], *PRE), pooled([AGENCY], *POST)
    print(f"{AGENCY} before {ab:.2f}, after {aa:.2f}")
    print(f"  before and after         : {100 * (aa / ab - 1):+.1f} percent")
    print(f"  difference in differences: "
          f"{100 * ((aa / ab) / (control_after / control_before) - 1):+.1f} percent")
    print(f"  four agencies pooled     : {did:+.1f} percent")
    print("  the truth                : -12.0 percent")
    n = clean[(clean["agency_id"] == AGENCY) & (clean["year_month"] >= SETTLED)]["n_uof"].sum()
    print(f"\n  incidents behind the post period estimate: {int(n)}")
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A004"
```

Millgate on its own gives **minus 18.7 percent** against a pooled estimate of
minus 11.3 and a truth of minus 12. Its before and after figure is minus 31.2.
The whole post programme estimate rests on **181 incidents**.

The method is not the problem. **A 12 percent effect is simply not measurable
from one small agency**, which is
[Module 4](Module_04_Why_Small_Agencies_Look_Volatile.ipynb) arriving for the
last time in this series. Pooling the trained agencies is what makes the
estimate usable, and it is also why an evaluation should be designed with
enough agencies in it before the programme launches rather than assembled
afterwards from whoever happened to adopt it.

</details>

---

**This is the end of the Intermediate series.**

You can now build a series from records, repair its calendar, choose and defend
a denominator, split it into trend and season, measure the trend with an
interval, adjust it, set limits on it, relate it to its own past and to other
series, construct a fair comparison group, forecast it, score the forecast
honestly, and estimate whether something changed.

**Next:** the [Advanced series](../../Advanced/) fits and defends formal
models, and the [Causal Inference series](../../../Causal_Inference/) takes up
the question this module had to leave open.

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*